In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

In [ ]:
#%cd ../..
#!ls

/Users/grushinda/Документы /Учеба/МИФИ/Классическое машинное обучение/HomeWorkers/Курсовая работа/IC50_CC50_SI_PROJ_GrushinDA
EDA           Models        catboost_info


In [6]:
tmp_data = pd.read_csv('EDA/data/findata-2.csv', index_col=0)

In [7]:
tmp_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   int64  
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   float6

In [8]:
go_data = tmp_data.copy()

In [9]:
# Удалим не нужные тут таргеты 
go_data['bin_target'] = (go_data['SI'] > 8).astype(int)

go_data = go_data.drop(columns=['IC50, mM','CC50, mM', 'SI'])


In [10]:
# Выбираем самые полезные параметры 

# Рассчитываем корреляцию всех признаков 
go_correlations = go_data.corr()['bin_target'].abs().sort_values()

# Отбираем признаки с корреляцией больше 0.1 
gl_high_info_features = go_correlations[go_correlations > 0.1]

print("Информативные признаки (есть связь):")
gl_high_info_features = gl_high_info_features.drop(['bin_target'], errors='ignore')


#Для финальной модели оставляем только информативные параметры  
gl_final_param = gl_high_info_features.index.unique().tolist() 

print(len(gl_final_param))
display(go_correlations.sort_values(ascending=False).head(50))

display(gl_final_param)

Информативные признаки (есть связь):
19


bin_target             1.000000
FractionCSP3           0.202072
SlogP_VSA6             0.182614
MaxAbsEStateIndex      0.154363
MaxEStateIndex         0.154363
HallKierAlpha          0.143002
VSA_EState8            0.139737
RingCount              0.132636
VSA_EState2            0.129195
BertzCT                0.126381
MinEStateIndex         0.122576
VSA_EState5            0.122553
VSA_EState4            0.118647
SMR_VSA1               0.116911
SMR_VSA10              0.114225
SMR_VSA5               0.111062
MaxPartialCharge       0.109826
NumHeteroatoms         0.103990
NumAliphaticRings      0.102778
SlogP_VSA5             0.101907
MinAbsPartialCharge    0.096755
BCUT2D_MWLOW           0.094602
NumRotatableBonds      0.093938
HeavyAtomMolWt         0.090279
SPS                    0.082955
FpDensityMorgan1       0.082035
BCUT2D_LOGPHI          0.081924
Chi1                   0.081693
NumHAcceptors          0.078527
VSA_EState7            0.078526
HeavyAtomCount         0.078431
ExactMol

['SlogP_VSA5',
 'NumAliphaticRings',
 'NumHeteroatoms',
 'MaxPartialCharge',
 'SMR_VSA5',
 'SMR_VSA10',
 'SMR_VSA1',
 'VSA_EState4',
 'VSA_EState5',
 'MinEStateIndex',
 'BertzCT',
 'VSA_EState2',
 'RingCount',
 'VSA_EState8',
 'HallKierAlpha',
 'MaxAbsEStateIndex',
 'MaxEStateIndex',
 'SlogP_VSA6',
 'FractionCSP3']

In [11]:
#перебором выявим параметры котрые коррелируют между собой > 60% и оставим только второй 

for col_1 in gl_final_param:
    for col_2 in gl_final_param:
        if col_1 != col_2:
            lv_correlation = go_data[col_1].corr(go_data[col_2])
            if lv_correlation >= 0.70:
                print(f' Параметр {col_1} коррелирует с парамтером {col_2} : {lv_correlation}')
                gl_final_param.remove(col_2)
display(gl_final_param)

 Параметр SlogP_VSA5 коррелирует с парамтером SMR_VSA5 : 0.8633308881356346
 Параметр SlogP_VSA5 коррелирует с парамтером VSA_EState8 : 0.7262475324148162
 Параметр NumHeteroatoms коррелирует с парамтером SMR_VSA1 : 0.7124995729362024
 Параметр NumHeteroatoms коррелирует с парамтером BertzCT : 0.7328144524665532
 Параметр MaxPartialCharge коррелирует с парамтером MaxAbsEStateIndex : 0.7372088681878393
 Параметр HallKierAlpha коррелирует с парамтером FractionCSP3 : 0.7974503576867145
 Параметр MaxEStateIndex коррелирует с парамтером MaxPartialCharge : 0.7372088681878393


['SlogP_VSA5',
 'NumAliphaticRings',
 'NumHeteroatoms',
 'SMR_VSA10',
 'VSA_EState4',
 'VSA_EState5',
 'MinEStateIndex',
 'VSA_EState2',
 'RingCount',
 'HallKierAlpha',
 'MaxEStateIndex',
 'SlogP_VSA6']

In [12]:
go_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1001 entries, 0 to 1000
Data columns (total 77 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MaxAbsEStateIndex    1001 non-null   float64
 1   MaxEStateIndex       1001 non-null   float64
 2   MinAbsEStateIndex    1001 non-null   float64
 3   MinEStateIndex       1001 non-null   float64
 4   qed                  1001 non-null   float64
 5   SPS                  1001 non-null   float64
 6   MolWt                1001 non-null   float64
 7   HeavyAtomMolWt       1001 non-null   float64
 8   ExactMolWt           1001 non-null   float64
 9   NumValenceElectrons  1001 non-null   int64  
 10  MaxPartialCharge     1001 non-null   float64
 11  MinPartialCharge     1001 non-null   float64
 12  MaxAbsPartialCharge  1001 non-null   float64
 13  MinAbsPartialCharge  1001 non-null   float64
 14  FpDensityMorgan1     1001 non-null   float64
 15  FpDensityMorgan2     1001 non-null   float6

In [20]:
print(go_data['bin_target'].value_counts())

bin_target
0    644
1    357
Name: count, dtype: int64


In [24]:
# Количество объектов для каждого класса (на уровне миноритарного класса)
K = go_data[go_data['bin_target'] == 1].shape[0]

# Сэмплируем K объектов для классов 0 
df0 = go_data[go_data['bin_target'] == 0].sample(K, random_state=42)

# Берем все объекты класса 1 (миноритарный класс)
df1 = go_data[go_data['bin_target'] == 1]

# Соединяем все вместе
go_data = pd.concat([df0, df1])

In [25]:
print(go_data['bin_target'].value_counts())

bin_target
0    357
1    357
Name: count, dtype: int64


In [26]:
X = go_data[gl_final_param]
#X = go_data.drop(columns='IC50, mM')
y = go_data['bin_target']

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (499, 12), (499,)
Test dataset size: (215, 12), (215,)


In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

models = {
    "LogisticRegression": (LogisticRegression(max_iter=1000), 
        {
            'C': [0.1, 1.0, 10.0], 
            'penalty': ['l2']
        }),

    "RandomForestClassifier": (RandomForestClassifier(), 
        {
            'n_estimators': [50, 150, 350],
            'max_depth': [None, 5, 10]
        }),

    "CatBoostClassifier": (CatBoostClassifier(), 
        {
            'iterations': [50, 150, 350],
            'learning_rate': [0.1, 0.5, 0.7],
            'verbose': [0]

        })
    
}



In [32]:
from skopt import BayesSearchCV
from sklearn.metrics import silhouette_score
from sklearn.model_selection import PredefinedSplit

# Перебор моделей
best_global_score = -10
best_model = None
results_report = []


for name, (model, params) in models.items():
    print(f"Обучаем {name}...")

    # Байесовская оптимизация гиперпараметров
    bayes_search = BayesSearchCV(
        estimator=model,
        search_spaces=params,
        n_iter=35,
        cv=15,
        scoring='roc_auc',  
        n_jobs=-1,
        random_state=42
    )

    # Обучение модели
    bayes_search.fit(X_train, y_train)

    # 
    score = bayes_search.best_score_  # type: ignore
    results_report.append({"Model": name, "Score": score, "Params": bayes_search.best_params_}) # type: ignore
    
    # Сохраняем абсолютного победителя
    if score > best_global_score:
        best_global_score = score
        best_model = bayes_search.best_estimator_ # type: ignore

# --- АНАЛИЗ ---
print("\n--- Report  ---")
print(pd.DataFrame(results_report))
print(f"\n Лучшая модель: {best_model}")



Обучаем LogisticRegression...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

Обучаем RandomForestClassifier...
Обучаем CatBoostClassifier...

--- Report  ---
                    Model     Score  \
0      LogisticRegression  0.668181   
1  RandomForestClassifier  0.747455   
2      CatBoostClassifier  0.735150   

                                              Params  
0                        {'C': 0.1, 'penalty': 'l2'}  
1           {'max_depth': None, 'n_estimators': 150}  
2  {'iterations': 350, 'learning_rate': 0.1, 'ver...  

 Лучшая модель: RandomForestClassifier(n_estimators=150)


In [97]:
# на тестовой выборке 
from sklearn import metrics

y_pred = best_model.predict(X_test)  # type: ignore

print("MAE", metrics.mean_absolute_error(y_test, y_pred))
print("MSE", metrics.mean_squared_error(y_test, y_pred))
print("R2 Score:", best_model.score(X_test, y_test)) # type: ignore

MAE 0.27906976744186046
MSE 0.27906976744186046
R2 Score: 0.7209302325581395
